## 1. 키 설정

In [1]:
import re, os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage


from dotenv import load_dotenv
load_dotenv()
#os.environ['OPENAI_API_KEY'] = ""

True

## 2. 시스템프롬프트 설정

In [8]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    request_timeout=60,
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
You are given a dialogue. Perform the following transformations:
1. Restore any omitted subjects, objects, or predicates. Example: change "고쳤어" → "[나는 창문을] 고쳤어".
2. Replace pronouns or vague references with specific nouns based on context. Example: change "이것" → "[사과]".
3. If the same speaker speaks consecutively, merge their utterances into one line so that the dialogue alternates between different speakers.
4. Do not transform unnecessary features.
Do not delete any sentence elements that do not fall under the above rules.

Mark only the modified parts with square brackets [ ].
Do not bracket unchanged words.
Keep the rest of the content exactly the same as the original except for the required modifications.



**Few-shot examples (mimic exactly this format)**

Example:
Input:
화자 2: 진짜 신의 한수
화자 1: 이사하자마자 비 많이 와서 베란다 물 많이 새는 거 알았잖아
화자 2: 글치 계속 해떴으면 몰랐겠지
화자 1: 그 때 물새는 거 알고 코킹작업해소 다행이다
화자 2: ㅇㅇ 안그랬으면 오늘처럼 비 많이 내리는 날 물바다됐을거야
화자 1: 요 아래 씽크홀 공사하던데 괜찮을라나
화자 2: 그러게 저번에도 비 많이 와서 땅꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물 안빠져서
화자 2: 새로 지은 곳인데도 그러네
화자 1: 부실공사지 뭐
화자 2: 비 많이 올 때는 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해
화자 1: 저번에 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 씽크홀 크기가 엄청나더라
화자 1: 오늘 비가 엄청 많이 내리네

Output:
화자 2: 진짜 [이번 상황은] 신의 한수
화자 1: [우리가] 이사하자마자 [장마철에] 비 많이 와서 [우리] 베란다 [창틀에서] 물 많이 새는 거 알았잖아
화자 2: [맞아, 그때] 계속 해떴으면 [물 새는 걸] 몰랐겠지
화자 1: [그 때 베란다에서] 물새는 거 알고 코킹작업해서 다행이다
화자 2: [응, 그때 코킹 안 했으면] 안 그랬으면 오늘처럼 비 많이 내리는 날 [베란다 안이] 물바다됐을 거야
화자 1: 요 아래 [아파트 단지 인근에서] 씽크홀 공사하던데 괜찮을라나
화자 2: 그러게 저번에도 비 많이 와서 [인근 도로가] 땅 꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 [빗물이] 안 빠져서
화자 2: [여기는] 새로 지은 곳인데도 그러네
화자 1: [이건 완전히] 부실공사지 뭐
화자 2: 비 많이 올 때는 [그 공사 구간]으로 다니지 말아야겠다
화자 1: 응 조심해. 저번에 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 씽크홀 크기가 엄청나더라
화자 1: 오늘 비가 엄청 많이 내리네

output only the transformed dialogue, nothing else. 
"""

integrated_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt.strip()),
    ("human", "<Actual dialogue to Process>\n{actual_dialogue}")
])

chain = integrated_prompt | llm

## 3. 실제 대화 데이터셋 수정해보기

### 3.1 nikluge-2025-대화 맥락 추론-train-000342

In [9]:
dialogue1 = '''
화자 1: 요즘 따라 여행가고 싶네요 ㅜㅜ
화자 2: 저도요ㅠㅠ
화자 2: 못나가서 더 그런걸까요?
화자 1: 맞아요 ㅜㅜ
화자 1: 코로나 지금 거의 2년 다되가잖아요
화자 1: 다들 여행이 근질근질 할거에요
화자 2: 맞아요ㅠ
화자 2: 해외는 진짜 꿈이에요
화자 1: 해외보단 국내가 그래도 좀 안전한 거 같아요
화자 2: 맞아요
화자 2: 바다가고싶어요!
화자 1: 동해바다 vs 서해바다
화자 2: 서해!
화자 1: 갯벌 좋아해요?
화자 2: 한번밖에 안가봤어요ㅠ
화자 2: 좋아하세요?
화자 1: 갯벌 진짜 꿀잼이에요 ㅋㅋㅋㅋ
화자 1: 거기서 게도 잡고 조개도 잡고~ 근데 옷 버릴 각오 해야되는 게 맴찢..
화자 2: 우와 가족끼리 같이 가고싶네요
화자 2: 버릴옷 필수,..
화자 1: 조카 있어요?
화자 1: 조카들이 많이 좋아할 것 같아요 ㅎㅎ
화자 2: 아직 조카가 없어요ㅜㅜ
화자 2: 아 애들이랑 같이가면 진짜 재밌겠네요
화자 1: 애들이랑 같이 가면 재밌는데ㅋㅋㅋ 자신을 포기해야해요
화자 1: 원주민 될 지도 몰라요 ㅎㅎ
'''
response1 = chain.invoke({"actual_dialogue": dialogue1})
print(response1.content.strip())

화자 1: 요즘 따라 여행 가고 싶네요 ㅜㅜ  
화자 2: 저도요ㅠㅠ 못 나가서 더 그런 걸까요?  
화자 1: 맞아요 ㅜㅜ 코로나 [시작된 지] 지금 거의 2년 다 돼 가잖아요. 다들 여행이 근질근질할 거예요  
화자 2: 맞아요ㅠ 해외는 진짜 꿈이에요  
화자 1: 해외보단 국내가 그래도 좀 안전한 거 같아요  
화자 2: 맞아요 바다 가고 싶어요!  
화자 1: 동해바다 vs 서해바다  
화자 2: 서해!  
화자 1: 갯벌 좋아해요?  
화자 2: [서해 갯벌에] 한 번밖에 안 가봤어요ㅠ 좋아하세요?  
화자 1: 갯벌 진짜 꿀잼이에요 ㅋㅋㅋㅋ 거기서 게도 잡고 조개도 잡고~ 근데 옷 버릴 각오 해야 되는 게 맴찢..  
화자 2: 우와 [저도] 가족끼리 같이 가고 싶네요. [갯벌에 갈 때는] 버릴 옷 필수,..  
화자 1: 조카 있어요? 조카들이 많이 좋아할 것 같아요 ㅎㅎ  
화자 2: 아직 조카가 없어요ㅜㅜ 아 [다른] 애들이랑 같이 가면 진짜 재밌겠네요  
화자 1: 애들이랑 같이 가면 재밌는데ㅋㅋㅋ [자신의 깔끔함을] 포기해야 해요. [갯벌에서] 원주민 될지도 몰라요 ㅎㅎ


### 3.2 nikluge-2025-대화 맥락 추론-train-000043

In [15]:
dialogue2 = '''
화자 1: 화장품 오래된거 쓰면 안돼요 ㅋ
화자 2: 당연한거가지고
화자 2: 피부 안좋아진다
화자 1: 맞아요 그니까 집에있는 클렌징오일도 버려요 ㅋ
화자 2: 그거 오래 안되지 않았어?
화자 1: 작년에 샀던가... 근데
화자 1: 피부에 뭐 나는거같아요 바르니까
화자 2: 두드러기 나면 안좋은데
화자 1: 그니까 버리는게 좋겠어요
화자 1: 저 틴트도 새로 사야할거같은데
화자 2: 그거 너무 빨갛더라
화자 1: 사진 보셨죠 ㅠ 완전 안맞음
화자 2: 쥐잡아먹은줄
화자 2: 알았다
화자 1: 그절도예요?? ㅋㅋㅋㅋ 그정돈아니었던거같은데
화자 2: 심했어
화자 1: 엄마두 화장 빨갛게 하시면서
화자 2: 그정도는 아니야 엄마는 잘하지
화자 1: 좋은데 코디가 가끔 별로세요
화자 2: 같이 보러가서 골라줘야지
화자 1: 근데 제가 골라두 별로 ㅠ 맘에 안들어하시는거같은데...
화자 2: 그래도 마음에 들어
'''
response2 = chain.invoke({"actual_dialogue": dialogue2})
print(response2.content.strip())

화자 1: [오래된] 화장품 [사용하면] 안 돼요 ㅋ  
화자 2: [그건] 당연한 거 가지고 피부 안 좋아진다  
화자 1: 맞아요 그니까 [우리] 집에 있는 클렌징 오일도 버려요 ㅋ  
화자 2: [클렌징 오일] 그거 오래 안 되지 않았어?  
화자 1: [그거] 작년에 샀던가... 근데 [그걸] 피부에 바르니까 뭐 나는 거 같아요  
화자 2: 두드러기 나면 안 좋은데  
화자 1: 그니까 [클렌징 오일] 버리는 게 좋겠어요. 저 틴트도 새로 사야 할 거 같은데  
화자 2: [네가 산] 그거 너무 빨갛더라  
화자 1: [내가 보낸] 사진 보셨죠 ㅠ 완전 안 맞음  
화자 2: [네가] 쥐 잡아먹은 줄 알았다  
화자 1: 그 정도예요?? ㅋㅋㅋㅋ 그 정도는 아니었던 거 같은데  
화자 2: [아니, 네가] 심했어  
화자 1: [우리] 엄마도 화장 빨갛게 하시면서  
화자 2: [엄마는] 그 정도는 아니야 엄마는 잘하지  
화자 1: [엄마 화장은] 좋은데 코디가 가끔 별로세요  
화자 2: [엄마랑] 같이 보러 가서 골라줘야지  
화자 1: 근데 제가 골라도 별로 ㅠ 맘에 안 들어 하시는 거 같은데...  
화자 2: 그래도 [엄마는 네가 고른 게] 마음에 들어


### 3.3 nikluge-2025-대화 맥락 추론-train-000095

In [16]:
dialogue3 = '''
화자 2: 그러게
화자 1: 이번주 토요일 모임잇는데
화자 2: 수~금 비온다는데
화자 1: 엉 근데 토요일은 맑음
화자 2: 금요일에 다시 봐야한다
화자 1: 응 안그래도 금요일 날씨보고 토요일 모임할지 최종결정
화자 2: 잘했다
화자 1: 토요일에도 더워서 야외에서 만나도 안추울듯
화자 2: 야외에서 볼려고
화자 1: 응 좀 그늘지고 한곳에 있으면 더워도 참을수 잇을듯하다
화자 2: 그래 선크링도 잘 바르고
화자 1: 응 양산도 가져가서 쓰고 있을려고 ㅋㅋ
화자 2: 모자 쓰면 괜찮을텐데
화자 1: 모자도 쓰고 이중으로 자외선 차단
화자 2: 대단하네
화자 1: 갑자기 날씨가 너무 추워짐
'''
response3 = chain.invoke({"actual_dialogue": dialogue3})
print(response3.content.strip())

화자 2: 그러게
화자 1: 이번 주 토요일 [친구들] 모임 있는데
화자 2: [수요일부터] 금요일까지 비 온다는데
화자 1: 응 근데 토요일은 맑음
화자 2: 금요일에 [날씨를] 다시 봐야 한다
화자 1: 응 안 그래도 금요일 [날] 날씨 보고 토요일 [친구들] 모임 할지 최종 결정
화자 2: 잘했다
화자 1: 토요일에도 더워서 야외에서 만나도 안 추울 듯
화자 2: [친구들] 야외에서 보려고
화자 1: 응 좀 그늘지고 한 곳에 있으면 더워도 참을 수 있을 듯하다
화자 2: 그래 [선크림도] 잘 바르고
화자 1: 응 양산도 가져가서 쓰고 있을려고 ㅋㅋ
화자 2: 모자 쓰면 괜찮을 텐데
화자 1: 모자도 쓰고 이중으로 자외선 차단
화자 2: 대단하네
화자 1: 갑자기 날씨가 너무 추워짐


### 3.4 nikluge-2025-대화 맥락 추론-train-000120

In [ ]:
dialogue4 = '''
화자 2: 오 정말? 무슨 일인데???
화자 1: 아 무슨 일인지 그동안 모랄ㅆ는데 이번에 일을줬어. 데이터 수집하고 웹사이트에 올리는거
화자 2: 빅데이터 관련 업무인가??낯선분야라 신기하다 일한지는 얼마나 됐어ㅓ??
화자 1: 빅데이터랑은 좀 다르더라 거긴 전문적이고 여긴 좀 덜 전문적 ㅋㅋ 이제 이주됐어
화자 2: 그렇구나 지금 한창 적응하고 잇겠네~ 일은 할 만해??
화자 1: 그동안 할만했는데 이제어려워졌었어..ㅠ
화자 2: 헉ㅠㅠ 앞날이 많이 힘들겠구나 머리가 아프겠는걸
화자 1: 그니까 어떠케 ㅠ 나 너무 바보같아서 아무것도 못하겟다능
화자 2: 아니야 처음하면 다들 똑같이 헤매지ㅠㅠ
화자 1: 마자..잘하기보다도 그냥..적당히만 하기라도 바랄뿐
화자 1: ㅠㅠ 난생 처음 보는 분야라너무 생소해..
화자 2: 너가 잘하길 응원할게
화자 2: 같이 일하는 동료들은 많아?
화자 1: 같이 5명 일해 나까지 5명  ㅎㅎㅎ...ㅎㅎㅎ...새롭다 새로워
화자 1: 너는 요새 일 괜찮아? 요새는ㅇ ㅓ때?
화자 2: 나는 늘 똑같지 뭐ㅎㅎ
'''
response4 = chain.invoke({"actual_dialogue": dialogue4})
print(response4.content.strip())

전체 데이터셋은 아래 구글 드리이브에서 확인 가능 </br>
https://docs.google.com/document/d/1LO62GYQokYjO8U68UilvwuQcANACik-o/edit?usp=sharing&ouid=101508814409893883423&rtpof=true&sd=true